# Atalaya · piloto acústico YAMNet

**Exploración técnica; no es un detector operativo.** El repositorio de
[DroneDetectionThesis v1.0.0](https://github.com/DroneDetectionThesis/Drone-detection-dataset/releases/tag/v1.0.0)
declara CC0, pero el [artículo de sus autores](https://www.mdpi.com/2504-446X/6/11/317)
indica que el vídeo visible incluye material externo; no está documentado si algún WAV procede
de esos vídeos. Se verificaron 24 clips por SHA-256 para una inspección inicial con YAMNet;
las nuevas descargas y el entrenamiento quedan **bloqueados** hasta aclarar derechos por
archivo, revisar voz y completar el intake. Los audios se pueden escuchar abajo para esa revisión.

El conjunto incluye 30 clips de dron, 30 de helicóptero y 30 de fondo. No hay IDs fiables de
vuelo, sesión o lugar por archivo. Cualquier reparto por clip sería solo una prueba de
funcionamiento, **no una medida de precisión en otro lugar ni de falsas alarmas de un
perímetro**. No se usa micrófono, audio de clientes, acceso al GitHub privado ni montaje de
Drive en el entorno de ejecución.

[Fraunhofer IDMT](https://zenodo.org/records/17525811) declara CC BY 4.0 y aporta una matriz
de ocho micrófonos para una prueba posterior de dirección de llegada. Una matriz única no
triangula posición por sí sola. [DADS](https://huggingface.co/datasets/geronimobasso/drone-audio-detection-samples)
queda excluido: pese a su etiqueta MIT, mezcla fuentes con restricciones de uso en seguridad,
defensa y uso comercial sin trazabilidad por muestra.


In [ ]:
# Dependencias para el entorno actual de Colab; se comprueba la versión de TensorFlow.
import importlib.metadata as meta
import subprocess, sys
tf_version = meta.version("tensorflow")
tf_minor = ".".join(tf_version.split(".")[:2])
print("TensorFlow de Colab:", tf_version)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "tensorflow-hub==0.16.1", f"tf-keras=={tf_minor}.*",
                       "librosa>=0.10.2,<0.12", "scikit-learn>=1.4,<2",
                       "soundfile>=0.12,<1"])
print("Dependencias preparadas. Si Colab pide reiniciar el entorno, reinícialo y vuelve a ejecutar.")


In [ ]:
import hashlib, json, random, urllib.request, wave
from pathlib import Path
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import tensorflow as tf
import tensorflow_hub as hub
from IPython.display import Audio, display

PILOT = json.loads('''{"source_commit":"e7a6eaf9760fa3f9ecc37d09cd11a8cd20be90dd","license_sha256":"a2010f343487d3f7618affe54f789f5487602331c0a8d03f49e9a7c547cf0499","clips":[{"name":"BACKGROUND_001.wav","sha256":"e8790309a7888c58c1d1f0f3bfa7850f193af5f23fceb8fdc830aad610ffeb28","bytes":1764052},{"name":"BACKGROUND_002.wav","sha256":"3bdef3845531382240df338a41e555d42023d98e8db63c54380fab82c23bde39","bytes":1764048},{"name":"BACKGROUND_003.wav","sha256":"7fcad6fffda1d56ba8c236480b04289e8c079ff518c0770770c0cb8c09c43e95","bytes":1764052},{"name":"BACKGROUND_004.wav","sha256":"8db1c6e9977126387f5ab00fe5d09cbda1a91e12d315a59356059d606eab0552","bytes":1764048},{"name":"BACKGROUND_005.wav","sha256":"1e86479957e5ef21e5ab4c4b25d3df1f09647a34957e621790f199cac8dfac28","bytes":1764048},{"name":"BACKGROUND_006.wav","sha256":"f73498b347bd77f4824cd42887ef72c08eb928e79ad2d2dba54f105d5b23fff5","bytes":1764052},{"name":"BACKGROUND_007.wav","sha256":"737b05e544fce41d5d2f2e608bae5fd199a6695a80f4c31d2e6945490f15c91d","bytes":1764048},{"name":"BACKGROUND_008.wav","sha256":"7f03a35e12c520b95eeab235ce7bec4d174b5adbf506e398da1f9d8a2f3f51a4","bytes":1764048},{"name":"BACKGROUND_009.wav","sha256":"e3cb52f2d2abef93cfe80724de76a96f4de8fa05597139e0ce5ad7dd32db5d7d","bytes":1764048},{"name":"BACKGROUND_010.wav","sha256":"37200cb91d6f12b4e5e6c79cfe42576252f033c67e862f737d7147ff52981318","bytes":1764048},{"name":"BACKGROUND_011.wav","sha256":"c59c17282d104ca1383efbe9addff87dd3c0669fd0a3edea6923f4fe86a4f9cc","bytes":1764052},{"name":"BACKGROUND_012.wav","sha256":"9dc31639da818626ff0f0995c793f1d36c12450ef5fd7e275dfc4fa9e52fe1f9","bytes":1764048},{"name":"BACKGROUND_013.wav","sha256":"ed96b240e24ad16fca1c430cccbed95754354567c9c1b4e707772504213864a7","bytes":1764048},{"name":"BACKGROUND_014.wav","sha256":"9bc49486d332ea8dc745fb6ec0c6cf46d5c20ebcb2bd43324d06996bd22fb680","bytes":1764048},{"name":"BACKGROUND_015.wav","sha256":"37c9ca48639871650a6610f43a08f78574c2cabb02ff69d84fcd4e6b9863c01c","bytes":1764052},{"name":"BACKGROUND_016.wav","sha256":"ed26ab3e21d613b3b2400e1fd93cb81435cca26e4ae02283460ada3ebcce754f","bytes":1764048},{"name":"BACKGROUND_017.wav","sha256":"c8e44c10e29e7f10452e1bfa228ceffee1ade9aaefd0e7f4e18a1f801af2b647","bytes":1764048},{"name":"BACKGROUND_018.wav","sha256":"0d618c8b488c2463de551890ed92499e56a2e203306e7d92ffef7ce20a409d69","bytes":1764052},{"name":"BACKGROUND_019.wav","sha256":"37200cb91d6f12b4e5e6c79cfe42576252f033c67e862f737d7147ff52981318","bytes":1764048},{"name":"BACKGROUND_020.wav","sha256":"d1bdd8d0fffa1e99b2a672160456e1eb2bd8c2b4ed41c99320dd982d576bd637","bytes":1764052},{"name":"BACKGROUND_021.wav","sha256":"88c517a8e97b1735716e2257ae5cd55f355ed5834f427c3746918e1b9411e778","bytes":1764052},{"name":"BACKGROUND_022.wav","sha256":"33537f39d3143b27d5a9de3c66e0f1e00c599b73b263989814ff9f17e59ec4f9","bytes":1764052},{"name":"BACKGROUND_023.wav","sha256":"2fce1d9350ab2b453676e156000ff292a520ba710bbc571e69e7b19aa51ebe4f","bytes":1764048},{"name":"BACKGROUND_024.wav","sha256":"991877ffaa560d376332a878afdeb1c95c080aee7e2a5d1ff0f7da44ecc93ffa","bytes":1764048},{"name":"BACKGROUND_025.wav","sha256":"c441558280f77504792a2a0c6a8829422a7763b366a45aab18ac381c7c43f066","bytes":1764048},{"name":"BACKGROUND_026.wav","sha256":"55654455627d97f252f30f63ea7f37e5ce7d9a6c076eb3580c630bc46e98a6e9","bytes":1764052},{"name":"BACKGROUND_027.wav","sha256":"9f4edfaca8194afbb264120cb002f7e9d4bceb03f0fc2ec39a88d6af2b251cd5","bytes":1764048},{"name":"BACKGROUND_028.wav","sha256":"af35ea6d7947b5d520054067233f932637f3852f1dc0f6e3e9479e525d9eac3a","bytes":1764048},{"name":"BACKGROUND_029.wav","sha256":"785c4bd25ef23a1a9f66af55accfb3bce6a06149c79bf9d236b8ce042562f397","bytes":1764052},{"name":"BACKGROUND_030.wav","sha256":"34f988beed58747337bf53bed8be7fcc3f0a6aadb06d49dc486fc48ef97d6b71","bytes":1764052},{"name":"DRONE_001.wav","sha256":"3c3fe29253e442cee58bb02b064e590e060d3e3e1c4039d8eb05d2da3b2c368c","bytes":1764048},{"name":"DRONE_002.wav","sha256":"a18e7825b0da4df748bea37d9392d92512192a7fe6447bea679d8980c8cfe3c4","bytes":1764052},{"name":"DRONE_003.wav","sha256":"12f04588de940bcd15ae955eb1b9a8a250986cf129f5c481f64598cd32cd302b","bytes":1764048},{"name":"DRONE_004.wav","sha256":"37c5062e4f07c9e05a67f7eb92a02785bddbf2fa10da8fd8596526755b54ec82","bytes":1764048},{"name":"DRONE_005.wav","sha256":"d56924abc9fe6ae53fae99e3eb5b72d2805c605581fa3e129eac8e578623705a","bytes":1764048},{"name":"DRONE_006.wav","sha256":"2079a59ea51d1e3468c29302d3c0c443b365fa27fe7dc53ac57a5fbc6d6b1edf","bytes":1764048},{"name":"DRONE_007.wav","sha256":"46bb93a90bec955129f6e2b2043d8d406fe93dedcd50a2a1519f82db1c59e420","bytes":1764048},{"name":"DRONE_008.wav","sha256":"3001968764aafe76b604c455a7464f19dde8bd5b91fc24f1f486be6df5daa749","bytes":1764048},{"name":"DRONE_009.wav","sha256":"b17bff5669ee90a509af546c0edf0ffb4703c26920b7137c71010edc9186f897","bytes":1764048},{"name":"DRONE_010.wav","sha256":"4798cb55d03757b17b4f43f40f9173eec0a0205de9797cce5424420c6833859c","bytes":1764048},{"name":"DRONE_011.wav","sha256":"d556d8297499b37f58ac90d0d3f865d553495688230905c52bd9c3019829cec9","bytes":1764048},{"name":"DRONE_012.wav","sha256":"7631a3915d5ac73cc704c12fef7de1c6020c0181c04f58be90318ec0f870b2d1","bytes":1764048},{"name":"DRONE_013.wav","sha256":"5548aac8a087f83faacd47da455af8a6e1f55489001200f222582ccdc81f7f08","bytes":1764052},{"name":"DRONE_014.wav","sha256":"94ace9424b53e69c34ca8184ab3bac397996807047d4abf3a31abf96abc22d9b","bytes":1764052},{"name":"DRONE_015.wav","sha256":"154de713d1e6bab713d91dedf66c623b313eb8dd74b9560fdc6199abc4f1d03e","bytes":1764052},{"name":"DRONE_016.wav","sha256":"c46e30be454f61aa0dad797daa4dddd4e747e893150eb0b99fc27ede5ee090da","bytes":1764052},{"name":"DRONE_017.wav","sha256":"789304d5fecdf52bee3eaea5b2b56ce20e67875b7d0df0b57bd15a7bf488719d","bytes":1764052},{"name":"DRONE_018.wav","sha256":"06c806bdbe8499a31976b9c63be89afddc35fc20ebceb5b7fef8d48105d22490","bytes":1764052},{"name":"DRONE_019.wav","sha256":"732ed57b672f7d1c6edc36f08fedad116119b4dd0d267c7e33ad20af6693acd0","bytes":1764052},{"name":"DRONE_020.wav","sha256":"68b73a293ca70e8791481492122cdc81db56c4967673c9f27d99844127bcb23d","bytes":1764052},{"name":"DRONE_021.wav","sha256":"7aba84d6e9edfa6766d18cc481022162e0c9a09734a502a6ac0789e8e9ce88a6","bytes":1764052},{"name":"DRONE_022.wav","sha256":"ddc68de1fdb7b5ec7477d897af55e1a3881991bbe1587c9d083844050e685df9","bytes":1764052},{"name":"DRONE_023.wav","sha256":"596be4f0dd105f98bc2b9a8fc0bbad69ba88645130f6cf54f683799555be53bf","bytes":1764052},{"name":"DRONE_024.wav","sha256":"719c315afc8b9aca1bc8428aeeed1306be1d8f381806e1c4ef89b2338f849b7d","bytes":1764052},{"name":"DRONE_025.wav","sha256":"4450b89707d09138e4aaf49b84df9d3188b3f7c778ba9af1fb149dec7534900a","bytes":1764052},{"name":"DRONE_026.wav","sha256":"a3fca5caa403979b87122f425db4c04fabd13c7d5505d2112c77d680870905a3","bytes":1764052},{"name":"DRONE_027.wav","sha256":"de55752be0ac29995d82cce0f0f9cedaa38a9c4405b0e180c27fed8030903fe5","bytes":1764052},{"name":"DRONE_028.wav","sha256":"cf401858d8fc05a969a2ad3389cfeffb9b824b96da344cc9216e37bce597f3bb","bytes":1764052},{"name":"DRONE_029.wav","sha256":"ca05e5b073ecc4e6567677d1a760f32ed96198c9f77b53ea90f07ba59915383b","bytes":1764048},{"name":"DRONE_030.wav","sha256":"9bcb1ee9e4e80086e238042a90929ea8c43a5549792f617e2669ef438dd50552","bytes":1764052},{"name":"HELICOPTER_001.wav","sha256":"ff04901b686c1b5aee3db7a7fcef94df677fd2e8cd49e6a840dc09e025260e90","bytes":1764048},{"name":"HELICOPTER_002.wav","sha256":"e9dbf544c189c0a1d80f3d599501432ecbd84e03ce5e1536114c6417d7de30d9","bytes":1764048},{"name":"HELICOPTER_003.wav","sha256":"976b0afd62fe6045673d37e0ba585979212603061c27bf9d1de4b23049b8f6af","bytes":1764048},{"name":"HELICOPTER_004.wav","sha256":"bdb7524eef645970587e1e96eaf23374255fc1789522f4bb69a42effc017d18b","bytes":1764048},{"name":"HELICOPTER_005.wav","sha256":"43961c8304d0de1d4dfb0ead9d97de8949a2e0893fc5bb3098ef9ab9a5e23120","bytes":1764048},{"name":"HELICOPTER_006.wav","sha256":"6ee524cabbff355ba4490ca975e284725531b59d2d660ef27ea0279215b95b08","bytes":1764048},{"name":"HELICOPTER_007.wav","sha256":"84102b6c5918a672ba8fb12750b7da9fd2d3f9857034627312f9d3a845610efc","bytes":1764048},{"name":"HELICOPTER_008.wav","sha256":"900639c6f5b02bbb3b42698550672aee7cd9b1fa77e8e9a8d00a0b0c1ed2e835","bytes":1764048},{"name":"HELICOPTER_009.wav","sha256":"7a69c14724199307229fb9fad26738ea4161a2734a8139cc865759a98718cc52","bytes":1764048},{"name":"HELICOPTER_010.wav","sha256":"a65b32fa367acfa63ce071b7e0b34b14e9aa82e571c2a5f96ae3b0067f8caa12","bytes":1764048},{"name":"HELICOPTER_011.wav","sha256":"cd9effd2f2552f29e6ae40fe6bf5f61ec9ead9cab6d428fbe242e939e9e50324","bytes":1764048},{"name":"HELICOPTER_012.wav","sha256":"94b8278d0c2ce8157250c229219505a8e18a1796d6d608de7a928eb896c19edb","bytes":1764048},{"name":"HELICOPTER_013.wav","sha256":"f4353777e7cafed31deef733eac90c65b0cc346e27503133acca5923ace0110e","bytes":1764052},{"name":"HELICOPTER_014.wav","sha256":"f0dbb7525b3dcb9d5269da1ef79ab1aaa94de912309ac4974caaacc75c8e103c","bytes":1764052},{"name":"HELICOPTER_015.wav","sha256":"a87b44b5cd1768bc657be7633c7ada7d93277d18c7214cfdbcbeb12c659f1cf4","bytes":1764052},{"name":"HELICOPTER_016.wav","sha256":"c68034426c7f0a97e4e95ebb7c7076893423835710aec2d7467c8b33b3dcef63","bytes":1764052},{"name":"HELICOPTER_017.wav","sha256":"7a8dcbbece59a44d276dc6b967ddf5b27a87a596d783eab3481f79510829f882","bytes":1764052},{"name":"HELICOPTER_018.wav","sha256":"769efc0d59b2107abb5e798a63781c43a1e382305f063f66d4f7f14a378a83b3","bytes":1764052},{"name":"HELICOPTER_019.wav","sha256":"f06f4a129f1ffe7d6ed9547f595d86ecf427e2d43a06c41f243d6a3ad8496ad4","bytes":1764052},{"name":"HELICOPTER_020.wav","sha256":"dec81d6f4386afda08ddb0129a2afc8a3c34f8239f30ffe58ddad9cd5cc585e0","bytes":1764052},{"name":"HELICOPTER_021.wav","sha256":"8677954e4aa82fe7db2281af1c7f6dbb064e82bc3f3fc66a04ccf23e2e571c05","bytes":1764052},{"name":"HELICOPTER_022.wav","sha256":"ff1f8395286e5cac5a24115a3b804480e1e0ebbc2fb903bae8c377ca0bd90fe5","bytes":1764052},{"name":"HELICOPTER_023.wav","sha256":"811f1e939c894a6f406d724f51bd6b06ba9dd271f03df3306ac456fc46af13a0","bytes":1764052},{"name":"HELICOPTER_024.wav","sha256":"d4b54c14bc09b136c9d7ce7bbd59ac27c6d34675c10bd675c7013d8294fdb3ba","bytes":1764052},{"name":"HELICOPTER_025.wav","sha256":"a82b468ae60f483a673b19794c8c43aefcea2bf66f7fa37fc533c918eb8fcb3d","bytes":1764052},{"name":"HELICOPTER_026.wav","sha256":"95184efe7292c1e9535a3f11580df0ccb0fff4a8536011690a5e738a1c393856","bytes":1764052},{"name":"HELICOPTER_027.wav","sha256":"a9555b719d5401d39c6713fe6dbb121298c1fab2015b29258a0143c9779a6fc0","bytes":1764052},{"name":"HELICOPTER_028.wav","sha256":"b4637a283e3175d935b9444cf29e5d47b1dec0736b45d365f517d3a3dc60af80","bytes":1764052},{"name":"HELICOPTER_029.wav","sha256":"00d4cb5e960aa0d613f412892993befb1b96f1c391709f8577d1a76cd47eb963","bytes":1764052},{"name":"HELICOPTER_030.wav","sha256":"55ecba3904aa9cf9427b6811a1ac69ae36e8c77c29c068742c5f971baf57f9cf","bytes":1764052}],"fraunhofer_record":"https://zenodo.org/api/records/17525811","dads_card":"https://huggingface.co/datasets/geronimobasso/drone-audio-detection-samples"}''')
SEED = 20261111
N_PER_CLASS = 8  # 24 WAV / ~42 MB; sube hasta 30 tras revisar el coste y la privacidad.
BASE = Path("/content/atalaya_piloto")
BASE.mkdir(parents=True, exist_ok=True)
COMMIT = PILOT["source_commit"]
print("Fuente fijada:", COMMIT, "| clips inventariados:", len(PILOT["clips"]))


In [ ]:
# Verifica la licencia original antes de descargar datos. Las redirecciones son restringidas.
ALLOWED_HOSTS = {"raw.githubusercontent.com", "api.github.com"}
def read_source(url):
    parsed = urlparse(url)
    assert parsed.scheme == "https" and parsed.hostname in ALLOWED_HOSTS, "Origen no permitido"
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Atalaya-Colab-Pilot/1"}), timeout=90) as r:
        redirected = urlparse(r.geturl())
        assert redirected.scheme == "https" and redirected.hostname in ALLOWED_HOSTS, "Redirección no permitida"
        return r.read()

ROOT_URL = f"https://raw.githubusercontent.com/DroneDetectionThesis/Drone-detection-dataset/{COMMIT}"
license_bytes = read_source(ROOT_URL + "/LICENSE")
assert hashlib.sha256(license_bytes).hexdigest() == PILOT["license_sha256"]
assert b"CC0 1.0 Universal" in license_bytes
print("Licencia CC0 original verificada por SHA-256.")


In [ ]:
# Selección reproducible y descarga con SHA-256 por archivo. Solo datos públicos.
# La atribución de cada WAV del artículo todavía no está probada. La descarga
# exploratoria inicial ya se verificó; las ejecuciones futuras quedan cerradas.
SOURCE_INTAKE_DOWNLOAD_APPROVED = False
assert SOURCE_INTAKE_DOWNLOAD_APPROVED, "Procedencia por WAV sin verificar: no repetir la ingesta."
assert 1 <= N_PER_CLASS <= 30
randomizer = random.Random(SEED)
selected = []
for prefix in ("BACKGROUND", "HELICOPTER", "DRONE"):
    candidates = sorted((c for c in PILOT["clips"] if c["name"].startswith(prefix + "_")),
                        key=lambda c: c["name"])
    assert len(candidates) == 30
    # La fuente contiene al menos un duplicado exacto: nunca meterlo dos veces.
    unique = {c["sha256"]: c for c in candidates}
    assert len(unique) >= N_PER_CLASS
    selected += randomizer.sample(sorted(unique.values(), key=lambda c: c["name"]), N_PER_CLASS)

for item in selected:
    name = item["name"]
    assert name.replace("_", "").replace(".", "").isalnum() and name.endswith(".wav")
    path = BASE / name
    if path.exists():
        raw = path.read_bytes()
    else:
        raw = read_source(f"{ROOT_URL}/Data/Audio/{name}")
    assert len(raw) == item["bytes"] and hashlib.sha256(raw).hexdigest() == item["sha256"], name
    if not path.exists():
        path.write_bytes(raw)
    with wave.open(str(path), "rb") as wav:
        assert wav.getframerate() == 44100 and wav.getsampwidth() == 2
        assert wav.getnchannels() in (1, 2) and 9.9 <= wav.getnframes()/44100 <= 10.1
print(f"Descargados y verificados {len(selected)} WAV ({sum(c['bytes'] for c in selected)/1e6:.1f} MB).")


## Revisión de privacidad

Escucha al menos una muestra aleatoria de cada clase y comprueba que no haya voz
identificable u otros datos personales. El resultado automático de YAMNet más abajo
sirve de ayuda, **no sustituye** esta escucha. Si aparece voz, no entrenes; documenta
el clip y resuelve su tratamiento antes de seguir. Los WAV permanecen solo en el
almacenamiento temporal de este entorno de Colab.


In [ ]:
# Reproducción de tres clips públicos, uno por clase, para revisión humana.
review_names = []
for prefix in ("BACKGROUND", "HELICOPTER", "DRONE"):
    candidate = next(c for c in selected if c["name"].startswith(prefix + "_"))
    review_names.append(candidate["name"])
    audio, sr = librosa.load(BASE / candidate["name"], sr=None, mono=True)
    print(candidate["name"], f"{len(audio)/sr:.1f} s")
    display(Audio(audio, rate=sr))


In [ ]:
# YAMNet oficial: señal mono float32 a 16 kHz; embeddings congelados por ventana.
MODEL_URL = "https://tfhub.dev/google/yamnet/1"
yamnet = hub.load(MODEL_URL)
class_map_path = yamnet.class_map_path().numpy().decode()
class_names = pd.read_csv(class_map_path)["display_name"].tolist()
assert len(class_names) == 521

rows, vectors, frame_vectors, waveforms = [], [], {}, {}
for item in selected:
    name = item["name"]
    signal, sr = librosa.load(BASE / name, sr=16000, mono=True)
    assert sr == 16000 and np.isfinite(signal).all()
    scores, embeddings, _ = yamnet(tf.convert_to_tensor(signal, dtype=tf.float32))
    score_array, embedding_array = scores.numpy(), embeddings.numpy()
    assert embedding_array.shape[1] == 1024
    vectors.append(embedding_array.mean(axis=0))
    frame_vectors[name] = embedding_array
    waveforms[name] = signal
    speech_columns = [i for i, label in enumerate(class_names)
                      if label in {"Speech", "Conversation", "Narration, monologue", "Child speech, kid speaking"}]
    speech_score = float(score_array[:, speech_columns].max()) if speech_columns else float("nan")
    rows.append({"clip": name, "clase": name.split("_")[0], "dron": int(name.startswith("DRONE_")),
                 "ventanas": len(score_array), "max_speech_score": round(speech_score, 3),
                 "evento_YAMNet": class_names[int(score_array.mean(axis=0).argmax())]})
df = pd.DataFrame(rows)
display(df.sort_values("max_speech_score", ascending=False).head(10))
print("Si YAMNet marca voz, revisa esos clips y no uses material con datos personales sin base resuelta.")


In [ ]:
# Intake pendiente: la ficha CC0 no identifica el origen de cada WAV.
# Esta celda documenta el bloqueo; no se desbloquea cambiando una casilla.
TRAINING_ALLOWED = False
assert TRAINING_ALLOWED, "Intake sin aprobar: derechos por fichero, revisión de voz, data card y agrupación."


In [ ]:
# Clasificador simple: regresión logística sobre embeddings congelados de YAMNet.
# Separación por clip: diagnóstica, NO evaluación independiente de vuelos o lugares.
assert TRAINING_ALLOWED, "Entrenamiento bloqueado: el intake de derechos, privacidad y grupos sigue pendiente."
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

X = np.asarray(vectors)
y = df["dron"].to_numpy()
idx = np.arange(len(y))
train_idx, test_idx = train_test_split(idx, test_size=0.25, random_state=SEED, stratify=y)
head = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
head.fit(X[train_idx], y[train_idx])
proba = head.predict_proba(X[test_idx])[:, 1]
pred = (proba >= 0.5).astype(int)
print("PRUEBA TÉCNICA POR CLIP · posible contaminación entre vuelos/sesiones · no sirve para campo")
print("train:", len(train_idx), "test:", len(test_idx), "umbral fijo: 0.5")
print(classification_report(y[test_idx], pred, labels=[0, 1], target_names=["no dron", "dron"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y[test_idx], pred, display_labels=["no dron", "dron"])
plt.title("Solo prueba técnica; generalización no verificada")
plt.show()


In [ ]:
# Resultado visual sobre un clip retenido: espectrograma y puntuación por ventana.
chosen = next((i for i in test_idx if y[i] == 1), test_idx[0])
name = df.iloc[chosen]["clip"]
signal = waveforms[name]
mel = librosa.feature.melspectrogram(y=signal, sr=16000, n_mels=64)
db = librosa.power_to_db(mel, ref=np.max)
window_scores = head.predict_proba(frame_vectors[name])[:, 1]
fig, (a, b) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
librosa.display.specshow(db, sr=16000, x_axis="time", y_axis="mel", ax=a)
a.set_title(f"{name} · espectrograma")
b.plot(np.arange(len(window_scores)) * 0.48, window_scores)
b.axhline(0.5, color="red", linestyle="--", label="umbral exploratorio")
b.set(ylim=(0, 1), ylabel="Puntuación", xlabel="Tiempo (s)")
b.legend()
fig.suptitle("Puntuación sin calibración; no usar para alertas reales")
plt.tight_layout(); plt.show()


## Fuentes para las siguientes pruebas

- **Fraunhofer IDMT:** [registro original de Zenodo](https://zenodo.org/records/17525811),
  licencia CC BY 4.0; 1,515,112,211 bytes de audio multicanal. Su estructura y
  coordenadas deben validarse antes de estimar dirección de llegada (azimut). Esta
  descarga grande queda fuera del ensayo rápido de clasificación.
- **DroneDetectionThesis:** el ensayo anterior incluye helicópteros reales como
  confusor. Su conjunto es pequeño y carece de grupos fiables de vuelo/sitio por WAV.
- **DADS:** la [ficha del autor](https://huggingface.co/datasets/geronimobasso/drone-audio-detection-samples)
  declara MIT pero cita [Al-Emadi](https://github.com/saraalemadi/DroneAudioDataset/blob/master/LICENSE.md),
  que prohíbe usos de seguridad y defensa, y [UrbanSound8K](https://urbansounddataset.weebly.com/),
  que limita a usos no comerciales. Hasta tener procedencia por archivo y derechos compatibles, `DADS_ENABLED=False`;
  ningún clip suyo se descarga ni se usa para entrenar este cuaderno.


In [ ]:
# Auditoría de metadatos de Fraunhofer; NO descarga su ZIP de 1.5 GB.
import urllib.request
FRAUNHOFER = PILOT["fraunhofer_record"]
with urllib.request.urlopen(FRAUNHOFER, timeout=30) as response:
    meta_record = json.load(response)
assert meta_record["metadata"]["license"]["id"] == "cc-by-4.0"
files = meta_record["files"]
assert len(files) == 1 and files[0]["key"] == "MicArrayData30s.zip"
assert files[0]["size"] == 1515112211
print("Fraunhofer:", meta_record["metadata"]["title"])
print("Licencia:", meta_record["metadata"]["license"]["id"], "| archivo:", files[0]["key"])
print("DOA/azimut: fase posterior, con control de geometría y referencia GPS.")
DADS_ENABLED = False
assert not DADS_ENABLED
print("DADS bloqueado hasta resolver licencias y procedencia por archivo.")


**Lectura de resultados:** cualquier acierto o error anterior describe solo
estos clips y este reparto. No se ha medido tasa de falsas alarmas por hora,
distancia de detección, precisión en cárceles, nuevos drones o emplazamientos.
Para eso se necesitan grabaciones de campo independientes, negativos continuos,
revisión de privacidad y una evaluación congelada por vuelo/sitio.
